# LLM-only placement vs the solver

**Exploration only.** Nothing here is part of the test suite and nothing here is authoritative:
the conclusions live in `documentation/failures.md`, and the numbers come from a run recorded in
`outputs/placement/` (Git-ignored). This notebook re-derives them from that file so they can be
poked at, not so they can be quoted.

The experiment is `SA-06`. It asks the model to do the solver's job — place the eleven demo boxes
as coordinates — ten times at temperature 0 and ten at temperature 1, and scores every reply with
`quai.checks.find_problems()`, the same independent check the solver answers to.

It makes no API calls. Re-run `python3 src/run_placement_experiment.py` for fresh replies.


In [ ]:
import json, pathlib, sys
sys.path.insert(0, '../src')

from demo import BOXES, VAN
from quai import llm_placement, solver
from quai.checks import find_problems

runs = sorted(pathlib.Path('../outputs/placement').glob('llm_placement_*.json'))
data = json.loads(runs[-1].read_text())
print(runs[-1].name, '|', data['model'], '|', data['runs_per_temperature'], 'runs per temperature')


## The solver, for comparison

Deterministic, and valid by construction — the interesting column is not whether it is correct.


In [ ]:
plan = solver.solve(BOXES, VAN)
print(f'placed {len(plan.placements)}/{len(BOXES)}  fill {plan.fill_rate:.1%}  problems {find_problems(plan.placements, VAN) or "none"}')
print('left out:', [b.id for b in plan.unplaced])


## Every reply, re-scored

`placed` is how many of the eleven the reply positioned; `fill` is the volume those boxes claim,
which is only meaningful when the plan is valid — an overlapping plan can claim any fill it likes.


In [ ]:
rows = []
for a in data['attempts']:
    att = llm_placement.score(a['raw'], BOXES, VAN)
    used = sum(p.dx * p.dy * p.dz for p in att.placements)
    rows.append({'t': a['temperature'], 'run': a['run'], 'parsed': att.parsed,
                 'placed': len(att.placements), 'fill': used / VAN.volume,
                 'problems': len(att.problems), 'valid': att.valid})

for r in rows:
    mark = 'VALID' if r['valid'] else f"{r['problems']} problems"
    print(f"t={r['t']:g} run {r['run']:2}  placed {r['placed']:2}/11  "
          f"fill {r['fill']:5.1%}  {mark}")


## How often is the answer the same answer?

Two plans are the same when every box sits at the same coordinates. The order of the list is not
a difference. Temperature 0 is the interesting row.


In [ ]:
from collections import Counter
for t in (0.0, 1.0):
    sigs = [llm_placement.score(a['raw'], BOXES, VAN).signature
            for a in data['attempts'] if a['temperature'] == t]
    counts = Counter(sigs)
    print(f't={t:g}: {len(counts)} distinct plans in {len(sigs)} runs; most common appeared {max(counts.values())} times')


## What goes wrong, by kind


In [ ]:
from collections import Counter
def kind(problem):
    for word, name in (('was not placed', 'not placed'), ('overlaps', 'overlap'),
                       ('outside', 'outside'), ('supported', 'unsupported'),
                       ('rotation', 'rotation')):
        if word in problem:
            return name
    return 'other'

for t in (0.0, 1.0):
    kinds = Counter()
    for a in data['attempts']:
        if a['temperature'] != t:
            continue
        for p in llm_placement.score(a['raw'], BOXES, VAN).problems:
            kinds[kind(p)] += 1
    print(f't={t:g}:', dict(kinds.most_common()))


## The one that worked

If a run produced a valid plan, this is it — and the comparison that matters is its fill against
the solver's, because a legal plan can still be a worse plan. Here it is not.


In [ ]:
valid = [a for a in data['attempts']
         if llm_placement.score(a['raw'], BOXES, VAN).valid]
print(f'{len(valid)} valid plan(s) in {len(data["attempts"])} runs')
for a in valid:
    att = llm_placement.score(a['raw'], BOXES, VAN)
    used = sum(p.dx * p.dy * p.dz for p in att.placements)
    print(f"  t={a['temperature']:g} run {a['run']}: {len(att.placements)}/11 placed, "
          f"fill {used / VAN.volume:.1%} against the solver's {plan.fill_rate:.1%}")
    for p in sorted(att.placements, key=lambda q: (q.z, q.x, q.y)):
        print(f'    {p.box.id:<10} ({p.x:>3},{p.y:>3},{p.z:>3})  {p.dx} x {p.dy} x {p.dz}')


## What to take from it

Written up properly in `documentation/failures.md`. In short: the model answers in the right shape
every time and is wrong about the geometry almost every time — and when it is right, it is better
than the solver, which says more about the greedy first-fit than about the model.
